## **신용카드 고객 세그먼트 분류 AI 경진대회**

### **1. 데이터**

고객별 금융 활동 데이터

* **Target 변수: 고객 세그먼트(A~E)**
* 회원 정보
* 신용 정보
* 승인매출정보
* 청구입금정보
* 잔액정보
* 채널정보
* 마케팅정보
* 성과정보



###**2. 코드 흐름**

**1. 데이터 로드 및 병합**

* 대용량 데이터로 인한 메모리 부족 방지를 위해 optimize_types() 함수로 다운캐스팅(float64->float32, int64->int32, object->category) 수행
* 데이터를 한 번에 병합하지 않고 순차적으로 병합 -> parquet 파일로 저장 후 재로드 -> del 과 gc.collect()로 메모리 누수 방지 및 메모리 효율 극대화

**2. 전처리 및 피처엔지니어링**

* 파생변수 생성(_최근3개월_연체율, _청구서혜택합계, _방문채널수_B0M, _총이용카드수 등)
* 결측률 80% 초과컬럼 및 nunique <= 1인 컬럼 제거
* 날짜형 변수를 '기준년월'과 차이로 계산해 '경과월' 변수로 변환 후 기존 날짜 컬럼 삭제

**3. 인코딩 및 클래스 불균형 처리**

* 인코딩: 타깃과 범주형 변수에 LabelEncoder 적용
* 클래스 불균형 처리: 클래스별 샘플 수에 반비례하는 Class Weight 계산

**4. 피처 선택**

* Class weight 적용한 XGBoost 모델로 gain 기준 Feature importance 산출
* custom_ratio 90% 이내에 해당하는 상위 353개 변수만 선택

**5. 모델링**

* Stratified 5-Fold 교차 검증으로 단일 XGBClassifier 모델 학습
* f1_macro 지표로 Early Stopping
* 5개 Fold 모델의 예측 확률 평균 후 최종 클래스 도출
* ID 당 6개월 치 예측 결과가 존재하므로 ID 별 최빈값(idxmax)로 집계


### **3. 주요 코드 분석**

**피처 중요도 산출**




In [ ]:
X_sample = X.sample(n=100000, random_state=42)
y_sample = y_encoded[X_sample.index]

sample_weights = np.array([weights[class_label] for class_label in y_sample])

temp_model = from xgboost import XGBClassifier
print(XGBClassifier.__module__)(
    device='cuda',
    enable_categorical=True,
    verbosity=0,
    random_state=42
)
temp_model.fit(X_sample,
               y_sample,
               sample_weight=sample_weights)


importance_dict = temp_model.get_booster().get_score(importance_type='gain')


if all(k in X.columns for k in importance_dict.keys()):
    importances = pd.Series(importance_dict)
else:
    # fallback: f0, f1 형식일 경우만 매핑
    feature_map = {f'f{idx}': col for idx, col in enumerate(X_sample.columns)}
    importances = pd.Series({feature_map.get(k, k): v for k, v in importance_dict.items()})

# 4. 메모리 정리
del temp_model
gc.collect()

* 데이터에서 10만 개만 무작위로 추출하고, 클래스 불균형을 보정하기 위해 미리 계산해 둔 가중치를 부여해 모델 학습
* gain을 기준으로 각 변수의 중요도를 가져옴. 변수명이 f0, f1 형태로 반환될 경우를 대비해 원래 컬럼을 매핑해 주는 else 구문이 포함됨
* 중요도 추출이 끝난 후 det temp_model과 gc.collect()를 호출해 메모리 정리

In [ ]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)

importance_df = pd.DataFrame(importances.items(), columns=['feature', 'importance'])
importance_df = importance_df.sort_values(by='importance', ascending=False).reset_index(drop=True)

importance_df['cumsum'] = importance_df['importance'].cumsum()
importance_df['cumsum_ratio'] = importance_df['cumsum'] / importance_df['importance'].sum()
gc.collect()

selected_features = importance_df[importance_df['cumsum_ratio'] <= 0.90]['feature'].tolist()
len(selected_features)

X_reduced = X[selected_features]
X_test = X_test[selected_features]
gc.collect()

* 누적 중요도(custom_ratio) 계산해 상위 90%에 해당하는 변수만 선택

**모델 앙상블**




In [ ]:
# 전체 model Ensemble
preds_proba_list = [model.predict_proba(X_test) for model in models]
avg_proba = np.mean(preds_proba_list, axis=0)
final_preds = np.argmax(avg_proba, axis=1)
y_test_pred_labels = le_target.inverse_transform(final_preds)

test_data = test.copy()  # 원본 유지
test_data["pred_label"] = y_test_pred_labels

* models 리스트에 담긴 각 Fold 모델이 예측한 클래스별 확률을 모두 구하고 평균을 구함
* 평균 확률이 가장 높은 클래스의 인덱스를 찾고, 숫자로 인코딩되어 있던 타깃 값을 le_target.inverse_transform으로 원래 세그먼트로 되돌림

### **4. 새롭게 알게 된 점/아쉬운 점**

**1. 새롭게 알게 된 점**

* 컬럼이 많은 데이터프레임 병합 시 데이터 타입 다운캐스팅과 중간 parquet 저장/재로딩이 메모리 관리에 효과적이라는 점을 배웠다.
* 고객별로 6개월 치 데이터가 존재할 때 각 시점별로 예측한 뒤 ID 단위로 최빈값을 취해 예측을 한 점이 인상적이다.

**2. 아쉬운 점**

* XGBoost 단일 모델만 사용했는데, 다른 트리 기반 모델과 앙상블하거나 튜닝을 했다면 더 높은 점수를 기대할 수도 있었을 것 같다.
* 피처 선택 시 일부 데이터만 사용했는데, 중요 변수가 탈락했을 위험이 있을 것 같다.  